# 🌾 PriceHarvest — EDA Analysis Notebook
**Project:** Crop Price Analysis and Trend Identification Using Historical Market Data  
**Team:** Sakshi Patil (51), Purva Pawar (54), Siddhi Shinde (68)  
**Institution:** SNDT Women's University, UMIT  
**Phase:** Phase 1 — EDA & Time-Series Analysis  

---

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams.update({
    'figure.facecolor': '#F8FAFC',
    'axes.facecolor':   '#F8FAFC',
    'axes.grid':        True,
    'grid.color':       '#E2E8F0',
    'font.family':      'DejaVu Sans',
})
PALETTE = ['#2563EB', '#16A34A', '#DC2626', '#D97706', '#7C3AED']
print('Libraries loaded successfully ✓')

## Step 2 — Load Datasets

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.preprocessing import load_all, combined
from src.features import engineer_all

# Load all 6 datasets
dfs = load_all()
dfs_fe = {k: engineer_all(df) for k, df in dfs.items()}
master = combined(dfs_fe)

print(f'\nMaster dataset: {master.shape[0]:,} rows × {master.shape[1]} columns')
print(f'Date range: {master["Arrival_Date"].min().date()} → {master["Arrival_Date"].max().date()}')

## Step 3 — Inspect the Data

In [ ]:
# Show first few rows
master[['State', 'Market', 'Commodity', 'Arrival_Date',
        'modal_price_kg', 'season', 'year', 'month_name']].head(10)

In [ ]:
# Check missing values
print('Missing values per column:')
print(master[['modal_price_kg', 'Arrival_Date', 'State', 'Market', 'Commodity']].isnull().sum())

## Step 4 — Summary Statistics

In [ ]:
master['Crop'] = master['Commodity'].str.title()
master['State_label'] = master['State'].apply(lambda s: 'Haryana' if 'Haryana' in s else 'Mumbai')

stats = master.groupby(['Crop', 'State_label'])['modal_price_kg'].agg(
    Count='count', Mean='mean', Median='median',
    Std='std', Min='min', Max='max'
).round(2)
stats['CV%'] = (stats['Std'] / stats['Mean'] * 100).round(1)
stats

## Step 5 — Price Distribution (Histogram + Box Plot)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Modal Price Distribution by Crop (₹/kg)', fontsize=15, fontweight='bold')

for ax, (crop, color) in zip(axes, [('Wheat', PALETTE[0]), ('Tomato', PALETTE[2]), ('Onion', PALETTE[1])]):
    data = master[master['Crop'] == crop]['modal_price_kg'].dropna()
    ax.hist(data, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(),   color='#DC2626', lw=2, ls='--', label=f'Mean ₹{data.mean():.2f}')
    ax.axvline(data.median(), color='#16A34A', lw=2, ls=':',  label=f'Median ₹{data.median():.2f}')
    ax.set_title(crop, fontsize=13, fontweight='bold')
    ax.set_xlabel('Modal Price (₹/kg)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/charts/eda_01_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6 — Monthly Price Trends (Time Series)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
fig.suptitle('Monthly Average Price Trend (2023–2025)', fontsize=15, fontweight='bold')

for ax, crop in zip(axes, ['Onion', 'Tomato', 'Wheat']):
    for state, color in [('Haryana', '#2563EB'), ('Mumbai', '#DC2626')]:
        sub = master[(master['Crop'] == crop) & (master['State_label'] == state)]
        ts = sub.groupby('Arrival_Date')['modal_price_kg'].mean() * 100  # back to ₹/Qtl
        monthly = ts.resample('M').mean()
        ax.plot(monthly.index, monthly.values, marker='o', markersize=3,
                lw=2, color=color, label=state)
        ax.fill_between(monthly.index, monthly.values, alpha=0.07, color=color)
    ax.set_title(f'{crop}', fontsize=12, fontweight='bold')
    ax.set_ylabel('₹/Quintal')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../outputs/charts/eda_02_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7 — Seasonal Patterns (Monthly Boxplots)

In [ ]:
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Monthly Price Box Plots by Crop', fontsize=15, fontweight='bold')

for ax, crop in zip(axes, ['Wheat', 'Tomato', 'Onion']):
    sub = master[master['Crop'] == crop]
    avail = [m for m in month_order if m in sub['month_name'].values]
    data  = [sub[sub['month_name'] == m]['modal_price_kg'].dropna().values for m in avail]
    bp = ax.boxplot(data, patch_artist=True, labels=avail,
                    medianprops=dict(color='#DC2626', lw=2))
    cols = plt.cm.Blues(np.linspace(0.3, 0.8, len(avail)))
    for patch, c in zip(bp['boxes'], cols):
        patch.set_facecolor(c)
    ax.set_title(crop, fontsize=12, fontweight='bold')
    ax.set_ylabel('Modal Price (₹/kg)')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../outputs/charts/eda_03_monthly_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8 — Haryana vs Mumbai Comparison (Bar Chart)

In [ ]:
avg = master.groupby(['Crop', 'State_label'])['modal_price_kg'].mean().unstack('State_label')
avg = avg * 100  # ₹/Quintal for display

fig, ax = plt.subplots(figsize=(10, 6))
avg.plot(kind='bar', ax=ax, color=['#2563EB', '#DC2626'], alpha=0.85,
         edgecolor='white', width=0.65)
ax.set_xlabel('Crop', fontsize=12)
ax.set_ylabel('Average Modal Price (₹/Quintal)', fontsize=12)
ax.set_title('Average Crop Price: Haryana vs Maharashtra', fontsize=14, fontweight='bold')
ax.legend(title='State', fontsize=11)
plt.xticks(rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt='₹%.0f', padding=3, fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/charts/eda_04_state_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — Price Volatility (Coefficient of Variation)

In [ ]:
vol = master.groupby(['Crop', 'State_label'])['modal_price_kg'].agg(
    mean='mean', std='std').assign(
    cv=lambda x: x['std'] / x['mean'] * 100).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
crops = ['Wheat', 'Tomato', 'Onion']
x = np.arange(len(crops))
w = 0.35
for i, (state, color) in enumerate([('Haryana', '#2563EB'), ('Mumbai', '#DC2626')]):
    vals = [vol[(vol['Crop'] == c) & (vol['State_label'] == state)]['cv'].values for c in crops]
    vals = [v[0] if len(v) else 0 for v in vals]
    bars = ax.bar(x + i * w, vals, w, label=state, color=color, alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.5,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x + w / 2)
ax.set_xticklabels(crops, fontsize=12)
ax.set_ylabel('Coefficient of Variation (%)')
ax.set_title('Price Volatility by Crop & State (CV%)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../outputs/charts/eda_05_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10 — Seasonal Heatmap (Year × Month)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Seasonal Price Heatmap (Month × Year) — All States', fontsize=14, fontweight='bold')
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

for ax, crop in zip(axes, ['Wheat', 'Tomato', 'Onion']):
    sub = master[master['Crop'] == crop]
    pivot = sub.groupby(['year', 'month'])['modal_price_kg'].mean().unstack(level=1) * 100
    pivot.columns = [month_names[m-1] for m in pivot.columns]
    sns.heatmap(pivot, ax=ax, cmap='YlOrRd', annot=True, fmt='.0f',
                linewidths=0.4, cbar_kws={'label': '₹/Qtl'})
    ax.set_title(crop, fontsize=12, fontweight='bold')
    ax.set_xlabel('Month')

plt.tight_layout()
plt.savefig('../outputs/charts/eda_06_seasonal_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 11 — Correlation Matrix (Min, Max, Modal Prices)

In [ ]:
price_cols = master[['min_price_kg', 'max_price_kg', 'modal_price_kg']].rename(
    columns={'min_price_kg': 'Min_Price', 'max_price_kg': 'Max_Price', 'modal_price_kg': 'Modal_Price'})
corr = price_cols.corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, ax=ax, cmap='Reds', annot=True, fmt='.2f',
            linewidths=0.5, vmin=0.5, vmax=1.0, square=True)
ax.set_title('Correlation: Min, Max, Modal Prices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/charts/eda_07_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key finding: Min ↔ Modal correlation =', round(corr.loc['Min_Price', 'Modal_Price'], 3))

## Step 12 — Key EDA Findings Summary

After completing EDA, we have answered these 5 core questions:

| Question | Finding |
|---|---|
| What is the average price range? | Wheat: ₹23–43/kg · Tomato: ₹22–24/kg · Onion: ₹19–21/kg |
| When are prices highest? | Tomato: Jul–Aug · Onion: Nov · Wheat: stable year-round |
| Where are prices different? | Wheat 84% costlier in Mumbai vs Haryana |
| How stable are prices? | Wheat CV≈5% (stable) · Tomato CV≈65–72% (volatile) |
| Do prices move together? | Min↔Modal correlation = 0.94 (highly correlated) |

---
**Phase 2** will extend this with ML models (Linear Regression, Random Forest, ARIMA) for price forecasting.